# module-extra-repr — ex1: extra_repr for a Linear-style module

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `module-extra-repr`. Running the final beacon cell reports progress against the `PyTorch: Module __repr__` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Module __repr__` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-extra-repr`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-extra-repr"
DD_SUBTOPIC = "PyTorch: Module __repr__"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `extra_repr` — quick refresher

`nn.Module.__repr__` is already implemented for you — it prints the class name and recursively the children's reprs. What it CAN'T know is which hyperparameters you stored on `self`. That's what `extra_repr` is for: override it to return a single string of `key=value` pairs that get inserted between the parentheses of the class repr.

```python
class Linear(nn.Module):
    def __init__(self, in_f, out_f, bias=True):
        super().__init__()
        self.in_features, self.out_features, self.has_bias = in_f, out_f, bias
        ...
    def extra_repr(self):
        return f'in_features={self.in_features}, out_features={self.out_features}, bias={self.has_bias}'
```

Then `print(Linear(3, 4))` shows `Linear(in_features=3, out_features=4, bias=True)` instead of the bare `Linear()`.

**Gotcha — print the bool, not the bias tensor.** ARENA's Linear exercise calls this out: `bias={self.bias}` would dump the full tensor; `bias={self.bias is not None}` prints the intended `True`/`False`.

### Exercise 1 — extra_repr for a Linear-style module

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Override extra_repr on an nn.Module subclass so that print(mod) displays in_features, out_features, and a bias-presence boolean (NOT the bias tensor).
> Keywords: extra_repr, __repr__, debugging, introspection
> ```

**KCs targeted:** `extra-repr-returns-string`, `extra-repr-print-bool-not-tensor`

Implement `MyReprLinear` — a Linear-style Module whose ONLY interesting feature is its `extra_repr`. In `__init__(self, in_features, out_features, bias=True)`:

1. Call `super().__init__()` first.
2. Store `self.in_features = in_features`, `self.out_features = out_features`.
3. Store `self.weight = nn.Parameter(t.zeros(out_features, in_features))` (zeros — initialization is not the point here).
4. If `bias` is True: store `self.bias = nn.Parameter(t.zeros(out_features))`. Otherwise: `self.bias = None`.
5. Implement `forward(self, x): return x @ self.weight.T + (self.bias if self.bias is not None else 0)`.
6. **Implement `extra_repr(self)`** returning the string `f'in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}'`.

**Critical:** the `bias=...` part must print the BOOLEAN `True` / `False` — NOT the bias tensor (`bias={self.bias}` would dump the whole tensor into the repr, which is ARENA's explicit gotcha for this atom).

Return an instance from `ex1_build_repr_linear(in_features, out_features, bias)`.

The test verifies that `repr(mod)` contains the right substrings and does NOT contain the bias tensor representation.

In [ ]:
def ex1_build_repr_linear(in_features: int, out_features: int, bias: bool = True):
    class MyReprLinear(t.nn.Module):
        def __init__(self, in_features, out_features, bias=True):
            super().__init__()
            self.in_features = in_features
            self.out_features = out_features
            self.weight = t.nn.Parameter(t.zeros(out_features, in_features))
            if bias:
                self.bias = t.nn.Parameter(t.zeros(out_features))
            else:
                self.bias = None
        def forward(self, x):
            return x @ self.weight.T + (self.bias if self.bias is not None else 0)
        def extra_repr(self):
            return f'in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}'
    return MyReprLinear(in_features, out_features, bias)


<details><summary>Solution</summary>

```python
def ex1_build_repr_linear(in_features: int, out_features: int, bias: bool = True):
    class MyReprLinear(t.nn.Module):
        def __init__(self, in_features, out_features, bias=True):
            super().__init__()
            self.in_features = in_features
            self.out_features = out_features
            self.weight = t.nn.Parameter(t.zeros(out_features, in_features))
            if bias:
                self.bias = t.nn.Parameter(t.zeros(out_features))
            else:
                self.bias = None
        def forward(self, x):
            return x @ self.weight.T + (self.bias if self.bias is not None else 0)
        def extra_repr(self):
            return f'in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}'
    return MyReprLinear(in_features, out_features, bias)
```

**Where extra_repr fits in the print output.** `nn.Module.__repr__` produces `ClassName(<extra_repr output>)` for leaf modules, or wraps the children's reprs inside for container modules. Your `extra_repr` only controls what goes between the parens — class name and children are handled for you.

**The bias-tensor-dump bug.** ARENA flags this explicitly. `f'bias={self.bias}'` formats the bias tensor via its own `__repr__`, producing output like `bias=Parameter containing: tensor([0., 0., 0., 0., 0.], requires_grad=True)` — useless for debugging Module shape issues. Always print the BOOLEAN presence flag (`self.bias is not None`), matching what `nn.Linear` itself does.

**Why `is not None` not `bool(self.bias)`.** Calling `bool(tensor)` on a multi-element tensor raises `RuntimeError: Boolean value of Tensor with more than one element is ambiguous`. The `is not None` check is the safe idiom.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()